# Component 7: Verification & Refinement

This notebook demonstrates the implementation of Component 7 (Boundary Critic, FP Auditor, and Refiner stub) as per the architectural specifications.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import sys
import os

# Ensure our src imports work
sys.path.append('..')

from src.components.component7_verification import Component7aBoundaryCritic, Component7bFPAuditor, Component7cRefiner

## 7a. Boundary Critic (ResNet18)
Evaluates the quality of predicted lesion boundaries.

In [ ]:
# Instantiate the model
critic = Component7aBoundaryCritic()
critic.eval()

# Verify parameter counts and frozen layers
frozen_params = sum(p.numel() for p in critic.parameters() if not p.requires_grad)
trainable_params = sum(p.numel() for p in critic.parameters() if p.requires_grad)

print(f"7a Boundary Critic (ResNet18):")
print(f"Frozen Parameters (Blocks 1-3): {frozen_params:,}")
print(f"Trainable Parameters (Block 4 + FC): {trainable_params:,}")

# Run a mock forward pass [B, 3, 224, 224]
mock_image_crop = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    bd_score = critic(mock_image_crop)

print(f"\nInput Shape: {mock_image_crop.shape}")
print(f"Output bd_score [1, 1] in [0,1]: {bd_score.item():.4f}")

## 7b. False Positive Auditor (DenseNet121)
Reuses the frozen DenseNet backbone to flag hard false positives.

In [ ]:
# Try to instantiate the model with torchxrayvision fallback if needed
try:
    auditor = Component7bFPAuditor(txh_fallback=True)
    print("Successfully loaded Component 7b FPAuditor")
except Exception as e:
    print(f"Error loading FPAuditor: {e}")

auditor.eval()

frozen_params_fp = sum(p.numel() for p in auditor.parameters() if not p.requires_grad)
trainable_params_fp = sum(p.numel() for p in auditor.parameters() if p.requires_grad)
print(f"\n7b FP Auditor (DenseNet121):")
print(f"Frozen Parameters (ALL Backbone): {frozen_params_fp:,}")
print(f"Trainable Parameters (FP Head MLP ONLY): {trainable_params_fp:,}") # Expected ~0.3M

# Run a mock forward pass using x_224 [B, 1, 224, 224]
mock_x224 = torch.randn(2, 1, 224, 224)
with torch.no_grad():
    fp_prob, concat_feats = auditor(mock_x224)
    
print(f"\nInput Shape: {mock_x224.shape}")
print(f"Concat features (1024 + 18): {concat_feats.shape}")
print(f"Output fp_prob: {fp_prob.shape} -> {fp_prob.squeeze().tolist()}")

## 7c. Refiner (MedSAM ViT-H Stub)
Logic for re-prompting MedSAM when Boundary score is low (< 0.7).

In [ ]:
refiner = Component7cRefiner()

# Mock inputs to test the branch logic
mock_img_embeds = torch.randn(2, 256, 64, 64)
mock_mask_fused = torch.ones(2, 1, 256, 256) * 0.5
mock_mask_var = torch.rand(2, 256, 256) # values 0 to 1
mock_bd_score = torch.tensor([[0.9], [0.4]]) # Batch 1: accepted, Batch 2: rejected/refine

print("Testing Refiner Logic Branch:")
print(f"bd_scores: {mock_bd_score.squeeze().tolist()}")

# Run refiner stub
refined_masks = refiner(mock_img_embeds, mock_mask_fused, mock_bd_score, mock_mask_var, None)
print(f"Returned Refined Masks Shape: {refined_masks.shape}")
print("Logic branching works. 7c relies entirely on zero-shot re-prompting.")